In [1]:
# High-performance VAE imputation - balanced optimization
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers, optimizers, regularizers, initializers, callbacks
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor, AdaBoostRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from scipy import stats
import warnings
import random
import os
import gc
import time

warnings.filterwarnings("ignore")

# Check the TensorFlow version
print(f"TensorFlow version: {tf.__version__}")

# ==================== Enhanced reproducibility settings ====================
SEED = 42

def set_all_seeds(seed=42):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# Initialize random seeds
set_all_seeds(SEED)

# GPU memory settings
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Found {len(gpus)} GPU devices")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("No GPU found. Running on the CPU.")

# Clear the session
tf.keras.backend.clear_session()
gc.collect()

# 1. Data loading and preprocessing
def load_and_preprocess_data():
    """Load the data and perform initial preprocessing."""
    data = pd.read_csv("./Earthquake.csv", low_memory=True)
    print(f"Original data shape: {data.shape}")
    
    # Keep a copy of the original data
    original_data = data.copy()
    
    # Drop nonnumeric columns
    cols_to_drop = []
    for col in data.columns:
        try:
            pd.to_numeric(data[col].dropna().iloc[:10])
        except:
            cols_to_drop.append(col)
    
    data = data.drop(columns=cols_to_drop)
    
    # Drop ID columns
    for col in data.columns[:3]:
        if data[col].dtype in ['int64', 'float64']:
            non_null = data[col].dropna()
            if len(non_null) > 1:
                diffs = non_null.diff().dropna()
                if (diffs == 1).mean() > 0.9:
                    data = data.drop(columns=[col])
    
    # Drop the L(m) column
    if 'L(m)' in data.columns:
        data = data.drop(columns=['L(m)'])
    
    # Drop columns with more than 50% missing values
    missing_ratio = data.isnull().sum() / len(data)
    cols_to_drop = missing_ratio[missing_ratio > 0.50].index
    data = data.drop(columns=cols_to_drop)
    
    # Convert data types
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce').astype('float32')
    
    print(f"Final data shape: {data.shape}")
    print(f"Features: {list(data.columns)}")
    
    return data, original_data

# 2. Balanced outlier treatment
def adaptive_outlier_handling(df):
    """Apply balanced outlier treatment."""
    df_clean = df.copy()
    transform_info = {}
    
    for column in df_clean.columns:
        mask = df_clean[column].notna()
        values = df_clean.loc[mask, column].values
        
        if len(values) == 0:
            continue
        
        # Calculate skewness and kurtosis
        skewness = stats.skew(values)
        kurtosis = stats.kurtosis(values)
        
        # Balanced outlier treatment
        if abs(skewness) > 2:  # Highly skewed
            # Use moderate quantile cutoffs
            lower, upper = np.percentile(values, [0.25, 99.75])
        else:  # Approximately normal
            # Use the standard IQR method
            q1, q3 = np.percentile(values, [25, 75])
            iqr = q3 - q1
            # Use an IQR multiplier of 2.75
            lower = q1 - 2.75 * iqr
            upper = q3 + 2.75 * iqr
            
            # Avoid discarding too much data
            lower = max(lower, np.percentile(values, 0.5))
            upper = min(upper, np.percentile(values, 99.5))
        
        # Soft clipping
        values_clipped = np.clip(values, lower, upper)
        
        transform_info[column] = {
            'type': 'balanced_clip',
            'lower': lower,
            'upper': upper,
            'skewness': skewness,
            'kurtosis': kurtosis
        }
        
        df_clean[column] = df_clean[column].astype(values_clipped.dtype, copy=False)
        df_clean.loc[mask, column] = values_clipped
    
    return df_clean, transform_info

# 3. Balanced scaling
def optimized_normalization(data):
    """Apply balanced scaling."""
    data_normalized = pd.DataFrame(index=data.index, columns=data.columns)
    scalers = {}
    
    for column in data.columns:
        mask = data[column].notna()
        values = data[column][mask].values.reshape(-1, 1)
        
        if len(values) > 0:
            # Calculate feature statistics
            skewness = stats.skew(values.flatten())
            value_range = values.max() - values.min()
            
            # Choose a scaling method
            if abs(skewness) > 1.5 or value_range > 1000:
                # Apply a quantile transform
                scaler = QuantileTransformer(
                    output_distribution='normal',  # Use a normal output distribution
                    n_quantiles=min(750, len(values)),  # Use a moderate number of quantiles
                    subsample=100000,
                    random_state=42
                )
            elif column in ['H(m)', 'At(m²）', 'Wa(m)']:
                # Use RobustScaler for selected features
                scaler = RobustScaler(quantile_range=(15, 85))  # Balanced quantile range
            else:
                # Use StandardScaler by default
                scaler = StandardScaler()
            
            scaler.fit(values)
            scaled = scaler.transform(values).flatten()
            
            # Map values to the balanced range [0.15, 0.85]
            min_val, max_val = scaled.min(), scaled.max()
            if max_val - min_val > 1e-8:
                scaled = (scaled - min_val) / (max_val - min_val) * 0.7 + 0.15
            else:
                scaled = np.ones_like(scaled) * 0.5
            
            data_normalized.loc[mask, column] = scaled.astype(np.float32)
            scalers[column] = {
                'scaler': scaler,
                'min': min_val,
                'max': max_val,
                'type': type(scaler).__name__
            }
    
    return data_normalized, scalers

# 4. Improved inverse scaling
def inverse_transform_optimized(data, scalers, column_names):
    """Reverse the scaling."""
    data_original = np.zeros_like(data)
    
    for col_idx, col_name in enumerate(column_names):
        values = data[:, col_idx].copy()
        
        if col_name in scalers:
            scaler_info = scalers[col_name]
            
            # Reverse the range mapping
            values = (values - 0.15) / 0.7
            values = values * (scaler_info['max'] - scaler_info['min']) + scaler_info['min']
            
            # Reverse the scaling
            values = scaler_info['scaler'].inverse_transform(values.reshape(-1, 1)).flatten()
        
        data_original[:, col_idx] = values
    
    return data_original

# 5. Balanced feature engineering
def enhanced_feature_engineering(data, mask):
    """Construct additional features."""
    n_samples, n_features = data.shape
    
    # Base features
    features = [data]
    
    # Add square-root features
    sqrt_data = np.sqrt(np.abs(data) + 1e-8) * np.sign(data)
    features.append(sqrt_data)
    
    # Add a few interaction terms using only three features
    if n_features <= 3:
        for i in range(n_features):
            for j in range(i+1, min(i+2, n_features)):
                interaction = data[:, i:i+1] * data[:, j:j+1]
                features.append(interaction)
    
    # Combine all features
    enhanced_data = np.hstack(features)
    
    # Expand the mask
    n_enhanced = enhanced_data.shape[1]
    n_base = data.shape[1]
    enhanced_mask = np.tile(mask, (1, n_enhanced // n_base))
    if enhanced_mask.shape[1] < n_enhanced:
        enhanced_mask = np.hstack([enhanced_mask, mask[:, :n_enhanced - enhanced_mask.shape[1]]])
    
    return enhanced_data, enhanced_mask

# 6. Balanced VAE model
class OptimizedVAE:
    """VAE with balanced regularization."""
    def __init__(self, data_dim, enhanced_dim, latent_dim=7, learning_rate=0.0006, seed=42):
        self.data_dim = data_dim
        self.enhanced_dim = enhanced_dim
        self.latent_dim = latent_dim  # Latent dimension
        self.learning_rate = learning_rate  # Learning rate
        self.seed = seed
        
        tf.random.set_seed(seed)
        
        self.encoder = self._build_encoder()
        self.decoder = self._build_decoder()
        
        # Optimizer settings
        self.optimizer = optimizers.Adam(
            learning_rate=learning_rate,
            beta_1=0.9,
            beta_2=0.999,
            epsilon=1e-7
        )
        
        self.best_weights = None
        self.best_val_loss = float('inf')
        self.training_history = {'train_loss': [], 'val_loss': []}
    
    def _build_encoder(self):
        """Build the balanced encoder."""
        inputs = layers.Input(shape=(self.enhanced_dim,))
        
        # Balanced network architecture
        x = layers.Dense(20, activation='relu',
                        kernel_initializer=initializers.HeNormal(seed=self.seed),
                        kernel_regularizer=regularizers.L2(0.015))(inputs)  # Moderate regularization
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.35)(x)  # Moderate dropout
        
        x = layers.Dense(14, activation='relu',
                        kernel_initializer=initializers.HeNormal(seed=self.seed),
                        kernel_regularizer=regularizers.L2(0.015))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.35)(x)
        
        # Latent space
        z_mean = layers.Dense(self.latent_dim,
                             kernel_regularizer=regularizers.L2(0.01))(x)
        z_log_var = layers.Dense(self.latent_dim,
                                kernel_regularizer=regularizers.L2(0.01))(x)
        
        # Limit the variance
        z_log_var = layers.Lambda(lambda x: tf.clip_by_value(x, -2.5, 1.5))(z_log_var)
        
        return keras.Model(inputs, [z_mean, z_log_var])
    
    def _build_decoder(self):
        """Build the balanced decoder."""
        inputs = layers.Input(shape=(self.latent_dim,))
        
        x = layers.Dense(14, activation='relu',
                        kernel_initializer=initializers.HeNormal(seed=self.seed),
                        kernel_regularizer=regularizers.L2(0.015))(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.35)(x)
        
        x = layers.Dense(20, activation='relu',
                        kernel_initializer=initializers.HeNormal(seed=self.seed),
                        kernel_regularizer=regularizers.L2(0.015))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.35)(x)
        
        # Output layer
        outputs = layers.Dense(self.data_dim, activation='sigmoid',
                              kernel_regularizer=regularizers.L2(0.008))(x)
        
        return keras.Model(inputs, outputs)
    
    def _sample(self, z_mean, z_log_var):
        """Sample using the reparameterization trick with moderate noise."""
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim), seed=self.seed)
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon * 0.12  # Moderate noise
    
    @tf.function
    def train_step(self, x_batch_enhanced, x_batch_original, mask_batch, beta):
        """Run one training step."""
        with tf.GradientTape() as tape:
            z_mean, z_log_var = self.encoder(x_batch_enhanced, training=True)
            z = self._sample(z_mean, z_log_var)
            reconstructed = self.decoder(z, training=True)
            
            # Combine MSE and Huber losses
            diff = x_batch_original - reconstructed
            
            # MSE component (weight 0.6)
            mse_loss = tf.square(diff) * mask_batch
            
            # Huber component (weight 0.4)
            huber_delta = 0.5
            is_small_error = tf.abs(diff) <= huber_delta
            small_error_loss = tf.square(diff) / 2
            large_error_loss = huber_delta * (tf.abs(diff) - huber_delta / 2)
            huber_loss = tf.where(is_small_error, small_error_loss, large_error_loss) * mask_batch
            
            # Combined reconstruction loss
            reconstruction_loss = (0.6 * tf.reduce_sum(mse_loss) + 
                                 0.4 * tf.reduce_sum(huber_loss)) / tf.reduce_sum(mask_batch)
            
            # Weighted KL loss
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )
            
            # Total loss
            total_loss = reconstruction_loss + beta * kl_loss
            
            # L2 regularization
            l2_loss = tf.add_n(self.encoder.losses + self.decoder.losses)
            total_loss += l2_loss
        
        variables = self.encoder.trainable_variables + self.decoder.trainable_variables
        gradients = tape.gradient(total_loss, variables)
        
        # Gradient clipping
        gradients = [tf.clip_by_norm(g, 0.75) for g in gradients]
        self.optimizer.apply_gradients(zip(gradients, variables))
        
        return total_loss, reconstruction_loss, kl_loss
    
    def fit(self, data_enhanced, data_original, mask, validation_split=0.2,
            epochs=90, batch_size=96):  # Balanced training settings
        """Train the model."""
        n_samples = len(data_original)
        indices = np.arange(n_samples)
        np.random.seed(self.seed)
        np.random.shuffle(indices)
        
        split_point = int(n_samples * (1 - validation_split))
        train_indices = indices[:split_point]
        val_indices = indices[split_point:]
        
        x_train_enhanced = data_enhanced[train_indices]
        x_train_original = data_original[train_indices]
        x_val_enhanced = data_enhanced[val_indices]
        x_val_original = data_original[val_indices]
        mask_train = mask[train_indices]
        mask_val = mask[val_indices]
        
        print(f"Training set: {len(x_train_original)} samples")
        print(f"Validation set: {len(x_val_original)} samples")
        
        patience = 18  # Early-stopping patience
        patience_counter = 0
        min_improvement = 0.0003  # Minimum improvement
        
        # KL annealing
        kl_anneal_epochs = 25
        
        for epoch in range(epochs):
            # Balanced KL annealing
            beta = min(0.8, epoch / kl_anneal_epochs) * 0.008
            
            train_indices_shuffled = np.random.permutation(len(x_train_original))
            train_loss = 0
            train_recon_loss = 0
            train_kl_loss = 0
            n_batches = 0
            
            for i in range(0, len(x_train_original), batch_size):
                batch_indices = train_indices_shuffled[i:i+batch_size]
                loss, recon_loss, kl_loss = self.train_step(
                    x_train_enhanced[batch_indices],
                    x_train_original[batch_indices],
                    mask_train[batch_indices],
                    beta
                )
                train_loss += loss
                train_recon_loss += recon_loss
                train_kl_loss += kl_loss
                n_batches += 1
            
            train_loss /= n_batches
            train_recon_loss /= n_batches
            train_kl_loss /= n_batches
            self.training_history['train_loss'].append(float(train_loss))
            
            # Validation
            if epoch % 5 == 0:
                val_loss = 0
                n_val_batches = 0
                
                for i in range(0, len(x_val_original), batch_size):
                    z_mean, z_log_var = self.encoder(x_val_enhanced[i:i+batch_size], training=False)
                    z = self._sample(z_mean, z_log_var)
                    reconstructed = self.decoder(z, training=False)
                    
                    diff = x_val_original[i:i+batch_size] - reconstructed
                    mse = tf.square(diff) * mask_val[i:i+batch_size]
                    val_loss += tf.reduce_sum(mse) / tf.reduce_sum(mask_val[i:i+batch_size])
                    n_val_batches += 1
                
                val_loss /= n_val_batches
                self.training_history['val_loss'].append(float(val_loss))
                
                # Early stopping
                if val_loss < self.best_val_loss - min_improvement:
                    self.best_val_loss = val_loss
                    self.best_weights = {
                        'encoder': self.encoder.get_weights(),
                        'decoder': self.decoder.get_weights()
                    }
                    patience_counter = 0
                else:
                    patience_counter += 5
                
                if epoch % 20 == 0:
                    print(f"Epoch {epoch}: Train Loss={train_loss:.6f}, "
                          f"Recon={train_recon_loss:.6f}, KL={train_kl_loss:.6f}, "
                          f"Val Loss={val_loss:.6f}")
                
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break
        
        # Restore the best weights
        if self.best_weights:
            self.encoder.set_weights(self.best_weights['encoder'])
            self.decoder.set_weights(self.best_weights['decoder'])
    
    def impute(self, data_enhanced, data_original, mask):
        """Impute missing values using a mix of latent means and samples."""
        imputed = np.zeros_like(data_original)
        batch_size = 192
        
        for i in range(0, len(data_original), batch_size):
            batch_enhanced = data_enhanced[i:i+batch_size]
            batch_original = data_original[i:i+batch_size]
            batch_mask = mask[i:i+batch_size]
            
            # Mixed sampling strategy
            predictions = []
            for _ in range(4):  # Four draws
                z_mean, z_log_var = self.encoder(batch_enhanced, training=False)
                # Use the latent mean for some passes and draw samples for the others
                if _ < 2:
                    z = z_mean  # Use the latent mean for the first two passes
                else:
                    z = self._sample(z_mean, z_log_var)  # Sample for the last two passes
                pred = self.decoder(z, training=False).numpy()
                predictions.append(pred)
            
            # Take a weighted average
            pred_mean = np.mean(predictions[:2], axis=0) * 0.6 + np.mean(predictions[2:], axis=0) * 0.4
            
            imputed[i:i+batch_size] = batch_original * batch_mask + pred_mean * (1 - batch_mask)
        
        return imputed

# 7. Balanced feature-specific models
class EnhancedTargetedModel:
    """Feature-specific models with balanced regularization."""
    def __init__(self, target_features, seed=42):
        self.target_features = target_features
        self.seed = seed
        self.base_models = {}
        
    def _get_model_for_feature(self, col_name):
        """Return the model configuration for a feature."""
        if col_name == 'H(m)':
            # Balanced ensemble
            models = [
                ('rf', RandomForestRegressor(
                    n_estimators=25,  # Number of trees
                    max_depth=4,  # Maximum tree depth
                    min_samples_split=25,
                    min_samples_leaf=12,
                    max_features=0.4,  # Fraction of features to consider
                    bootstrap=True,
                    random_state=self.seed
                )),
                ('ridge', Ridge(alpha=2.0, random_state=self.seed)),  # Regularization
                ('gbr', GradientBoostingRegressor(
                    n_estimators=20,
                    max_depth=3,
                    learning_rate=0.04,
                    random_state=self.seed
                ))
            ]
            return VotingRegressor(models)
            
        elif col_name == 'At(m²）':
            # Balanced random forest
            return RandomForestRegressor(
                n_estimators=30,
                max_depth=4,
                min_samples_split=30,
                min_samples_leaf=15,
                max_features=0.4,
                bootstrap=True,
                random_state=self.seed
            )
            
        else:  # Wa(m)
            # Balanced gradient boosting
            return GradientBoostingRegressor(
                n_estimators=25,
                max_depth=3,
                learning_rate=0.04,
                subsample=0.6,
                min_samples_split=25,
                min_samples_leaf=12,
                random_state=self.seed
            )
    
    def fit_and_impute(self, data_original, mask, column_names, train_indices=None):
        """Train on the available observations and impute missing values."""
        simple_imputer = SimpleImputer(strategy='median')
        
        if train_indices is not None:
            train_data = data_original[train_indices]
            simple_imputer.fit(train_data)
        else:
            simple_imputer.fit(data_original)
        
        data_filled = simple_imputer.transform(data_original)
        imputed = data_original.copy()
        
        for col_idx, col_name in enumerate(column_names):
            if col_name not in self.target_features:
                continue
            
            if train_indices is not None:
                train_mask = np.zeros(len(data_original), dtype=bool)
                train_mask[train_indices] = True
                obs_mask = (mask[:, col_idx] == 1) & train_mask
            else:
                obs_mask = mask[:, col_idx] == 1
            
            if np.sum(obs_mask) < 50:
                continue
            
            # Train the model
            X_train = np.delete(data_filled[obs_mask], col_idx, axis=1)
            y_train = data_original[obs_mask, col_idx]
            
            # Regularize with noise
            noise = np.random.normal(0, 0.008, X_train.shape) * np.std(X_train, axis=0)
            X_train_noisy = X_train + noise
            
            model = self._get_model_for_feature(col_name)
            model.fit(X_train_noisy, y_train)
            
            # Predict missing values
            miss_mask = mask[:, col_idx] == 0
            if np.sum(miss_mask) > 0:
                X_miss = np.delete(data_filled[miss_mask], col_idx, axis=1)
                
                # Average multiple predictions
                predictions = []
                for _ in range(3):
                    noise = np.random.normal(0, 0.004, X_miss.shape) * np.std(X_miss, axis=0)
                    predictions.append(model.predict(X_miss + noise))
                
                imputed[miss_mask, col_idx] = np.mean(predictions, axis=0)
        
        return imputed

# 8. Final balanced ensemble
class FinalEnsemble:
    """Final ensemble with balanced weights."""
    def __init__(self, data_dim, seed=42):
        self.data_dim = data_dim
        self.seed = seed
        self.models = []
        self.weights = None
        self.scalers = None
        self.data_raw = None
        self.column_names = None
    
    def fit(self, data_normalized, mask, data_raw, scalers, column_names):
        """Train the final ensemble."""
        print("\nTraining the final ensemble...")
        self.scalers = scalers
        self.data_raw = data_raw
        self.column_names = column_names
        
        # Construct additional features
        data_enhanced, mask_enhanced = enhanced_feature_engineering(data_normalized, mask)
        print(f"Expanded feature count: {data_enhanced.shape[1]}")
        
        # 1. KNN imputer
        print("1. KNN imputer...")
        knn = KNNImputer(n_neighbors=6, weights='distance')  # Number of neighbors
        knn.fit(data_normalized)
        self.models.append(('knn', knn))
        
        # 2. IterativeImputer
        print("2. IterativeImputer...")
        mice = IterativeImputer(
            estimator=BayesianRidge(),  # Use Bayesian ridge regression
            max_iter=4,  # Number of iterations
            random_state=self.seed
        )
        mice.fit(data_normalized)
        self.models.append(('mice', mice))
        
        # 3. Optimized VAE
        print("3. VAE model...")
        vae = OptimizedVAE(
            data_dim=self.data_dim,
            enhanced_dim=data_enhanced.shape[1],
            latent_dim=7,  # Latent dimension
            learning_rate=0.0006,  # Learning rate
            seed=self.seed
        )
        
        mean_imputer = SimpleImputer(strategy='median')
        mean_imputer.fit(data_normalized)
        initial_imputed = mean_imputer.transform(data_normalized)
        data_filled = np.where(mask == 1, data_normalized, initial_imputed)
        
        data_filled_enhanced, _ = enhanced_feature_engineering(data_filled, mask)
        
        vae.fit(data_filled_enhanced, data_filled, mask, 
                validation_split=0.2, epochs=90, batch_size=96)
        self.models.append(('vae', vae))
        self.vae_enhanced_data = data_filled_enhanced
        
        # 4. Feature-specific models
        print("4. Feature-specific models (trained on demand during prediction)...")
        target_features = [col for col in column_names if col in ['H(m)', 'At(m²）', 'Wa(m)']]
        if not target_features:
            target_features = column_names[:min(3, len(column_names))]
        
        self.target_model = EnhancedTargetedModel(
            target_features=target_features,
            seed=self.seed
        )
        self.models.append(('targeted', self.target_model))
        
        print("Ensemble training complete")
        
        # Adaptive weights
        self._compute_adaptive_weights()
    
    def _compute_adaptive_weights(self):
        """Calculate balanced adaptive weights."""
        print("\nCalculating adaptive weights...")
        
        # Balanced weight allocation
        base_weights = {
            'knn': 0.27,      # KNN weight
            'mice': 0.28,     # MICE weight
            'vae': 0.27,      # VAE weight
            'targeted': 0.18  # Feature-specific model weight
        }
        
        self.weights = np.array([base_weights[name] for name, _ in self.models])
        self.weights = self.weights / np.sum(self.weights)
        
        print(f"Final weights: {dict(zip([n for n, _ in self.models], self.weights))}")
    
    def impute(self, data_normalized, mask, column_names, train_indices=None):
        """Predict using the final ensemble."""
        predictions = []
        
        for (name, model), weight in zip(self.models, self.weights):
            if name == 'knn':
                pred = model.transform(data_normalized)
                pred_original = inverse_transform_optimized(pred, self.scalers, column_names)
                
            elif name == 'mice':
                pred = model.transform(data_normalized)
                pred_original = inverse_transform_optimized(pred, self.scalers, column_names)
                
            elif name == 'vae':
                mean_imputer = SimpleImputer(strategy='median')
                mean_imputer.fit(data_normalized)
                initial_imputed = mean_imputer.transform(data_normalized)
                data_filled = np.where(mask == 1, data_normalized, initial_imputed)
                data_filled_enhanced, _ = enhanced_feature_engineering(data_filled, mask)
                
                pred = model.impute(data_filled_enhanced, data_filled, mask)
                pred_original = inverse_transform_optimized(pred, self.scalers, column_names)
                
            elif name == 'targeted':
                pred_original = model.fit_and_impute(
                    self.data_raw, mask, column_names, train_indices
                )
            
            predictions.append(pred_original * weight)
        
        # Weighted average
        imputed = np.sum(predictions, axis=0)
        
        # Preserve observed values
        imputed = self.data_raw * mask + imputed * (1 - mask)
        
        # Light postprocessing
        for col_idx in range(imputed.shape[1]):
            miss_mask = mask[:, col_idx] == 0
            if np.sum(miss_mask) > 2:
                # Apply light smoothing
                from scipy.ndimage import gaussian_filter1d
                imputed[miss_mask, col_idx] = gaussian_filter1d(
                    imputed[miss_mask, col_idx], sigma=0.5
                )
        
        return imputed


# =================== Comprehensive evaluation ===================
def comprehensive_evaluation(model, data_normalized, mask, column_names, data_raw, scalers):
    print("\n=== Final evaluation (5-fold cross-validation) ===")
    results = []
    for col_idx, col_name in enumerate(column_names):
        obs_mask = mask[:, col_idx] == 1
        obs_indices = np.where(obs_mask)[0]
        if len(obs_indices) < 100:
            continue
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        all_true_original, all_pred_original, fold_metrics = [], [], []
        for fold, (train_idx, test_idx) in enumerate(kf.split(obs_indices)):
            train_indices = obs_indices[train_idx]
            test_indices = obs_indices[test_idx]
            test_mask = mask.copy()
            test_mask[test_indices, col_idx] = 0
            imputed_original = model.impute(data_normalized, test_mask, column_names, train_indices=train_indices)
            true_values = data_raw[test_indices, col_idx]
            pred_values = imputed_original[test_indices, col_idx]
            if len(true_values) > 1:
                fold_r2 = r2_score(true_values, pred_values)
                fold_mae = mean_absolute_error(true_values, pred_values)
                fold_metrics.append({'r2': fold_r2, 'mae': fold_mae})
            all_true_original.extend(true_values)
            all_pred_original.extend(pred_values)
        all_true_original = np.array(all_true_original)
        all_pred_original = np.array(all_pred_original)

        # ============== Export observed and predicted values to CSV ==============
        try:
            save_df = pd.DataFrame({
                "Index": np.arange(len(all_true_original)),
                "True_Value": all_true_original,
                "Predicted_Value": all_pred_original
            })
            safe_name = col_name.replace("/", "_").replace("（", "_").replace("）", "_")
            save_df.to_csv(f"observed_vs_predicted_{safe_name}.csv", index=False, encoding='utf-8-sig')
        except Exception as e:
            print(f"⚠️ Could not save predictions for {col_name}: {e}")
        # ============================================================

        mae = mean_absolute_error(all_true_original, all_pred_original)
        rmse = np.sqrt(mean_squared_error(all_true_original, all_pred_original))
        ss_res = np.sum((all_true_original - all_pred_original) ** 2)
        ss_tot = np.sum((all_true_original - np.mean(all_true_original)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        r2_scores = [m['r2'] for m in fold_metrics]
        r2_std = np.std(r2_scores) if r2_scores else 0
        r2_mean = np.mean(r2_scores) if r2_scores else r2
        non_zero_mask = np.abs(all_true_original) > np.percentile(np.abs(all_true_original), 5)
        if np.sum(non_zero_mask) > 0:
            ape_values = np.abs((all_true_original[non_zero_mask] - all_pred_original[non_zero_mask]) / all_true_original[non_zero_mask])
            mape = np.mean(np.clip(ape_values, 0, np.percentile(ape_values, 95))) * 100
        else:
            mape = np.mean(np.abs(all_true_original - all_pred_original)) * 100
        correlation = np.corrcoef(all_true_original, all_pred_original)[0, 1] if np.std(all_true_original) > 0 and np.std(all_pred_original) > 0 else 0
        status = "Excellent" if r2 >= 0.75 else "Good" if r2 >= 0.65 else "Acceptable" if r2 >= 0.55 else "Needs improvement"
        results.append({
            'Feature': col_name, 'CV Mean R²': r2_mean, 'CV Std': r2_std,
            'Overall R²': r2, 'Correlation': correlation, 'MAE': mae,
            'RMSE': rmse, 'MAPE(%)': mape, 'Status': status
        })
        print(f"{col_name}: CV mean R²={r2_mean:.4f}±{r2_std:.4f}, overall R²={r2:.4f}, MAPE={mape:.2f}%, status={status}")
    return pd.DataFrame(results)

# 10. Main function
def main():
    """Run the workflow."""
    print("=" * 80)
    print("High-performance VAE imputation - balanced optimization")
    print("Main improvements:")
    print("1. Balanced regularization")
    print("2. Optimized model complexity")
    print("3. Combined loss function")
    print("4. Balanced ensemble weights")
    print("5. Improved evaluation metrics")
    print("=" * 80)
    
    try:
        set_all_seeds(SEED)
        
        # 1. Load the data
        print("\n1. Loading data...")
        data_raw, original_data = load_and_preprocess_data()
        
        # 2. Adaptive outlier treatment
        print("\n2. Applying adaptive outlier treatment...")
        data_clean, transform_info = adaptive_outlier_handling(data_raw)
        
        # 3. Optimized scaling
        print("\n3. Applying optimized mixed scaling...")
        data_normalized, scalers = optimized_normalization(data_clean)
        
        # Create the mask
        mask = (~data_normalized.isnull()).astype(np.float32).values
        data_array = data_normalized.fillna(0.5).values.astype(np.float32)
        data_raw_array = data_clean.fillna(0).values.astype(np.float32)
        
        print(f"\nData shape: {data_array.shape}")
        print(f"Missing rate: {(1 - mask.mean())*100:.2f}%")
        
        # Report the scaling methods
        print("\nScaling methods:")
        for col_name, scaler_info in scalers.items():
            print(f"  {col_name}: {scaler_info['type']}")
        
        # 4. Train the final ensemble
        print("\n4. Training the final ensemble...")
        model = FinalEnsemble(data_dim=data_array.shape[1], seed=SEED)
        model.fit(data_array, mask, data_raw_array, scalers, data_normalized.columns)
        
        # 5. Evaluate the model
        print("\n5. Evaluating the model...")
        eval_results = comprehensive_evaluation(
            model, data_array, mask, data_normalized.columns,
            data_raw_array, scalers
        )
        
        # 6. Impute using all data
        print("\n6. Generating the final imputed data...")
        final_imputed_original = model.impute(data_array, mask, data_normalized.columns)
        
        # 7. Report the results
        print("\n" + "=" * 80)
        print("Final model performance (balanced optimization)")
        print("=" * 80)
        
        for _, row in eval_results.iterrows():
            print(f"\n{row['Feature']}:")
            print(f"  CV mean R² = {row['CV Mean R²']:.4f} ± {row['CV Std']:.4f}")
            print(f"  Overall R² = {row['Overall R²']:.4f}")
            print(f"  MAE = {row['MAE']:.4f}")
            print(f"  RMSE = {row['RMSE']:.4f}")
            print(f"  MAPE = {row['MAPE(%)']:.2f}%")
            print(f"  Status: {row['Status']}")
        
        # Summary statistics
        print("\n" + "-" * 40)
        print("Summary statistics")
        print("-" * 40)
        
        avg_cv_r2 = eval_results['CV Mean R²'].mean()
        avg_total_r2 = eval_results['Overall R²'].mean()
        avg_mape = eval_results['MAPE(%)'].mean()
        avg_cv_std = eval_results['CV Std'].mean()
        
        print(f"Mean CV R²: {avg_cv_r2:.4f} ± {avg_cv_std:.4f}")
        print(f"Mean overall R²: {avg_total_r2:.4f}")
        print(f"Mean MAPE: {avg_mape:.2f}%")
        
        # Stability analysis
        print(f"\nStability analysis:")
        for _, row in eval_results.iterrows():
            print(f"  {row['Feature']}: CV std={row['CV Std']:.4f}")
        
        # Save the results
        final_df_original = pd.DataFrame(final_imputed_original, columns=data_normalized.columns)
        final_df_original.to_csv("imputed_earthquake.csv", index=False)
        eval_results.to_csv("evaluation_earthquake.csv", index=False)
        
        print("\nFiles saved:")
        print("- imputed_earthquake.csv")
        print("- evaluation_earthquake.csv")
        
        print("\n" + "=" * 50)
        print("Final diagnostics")
        print("=" * 50)
        
        # Overall diagnostics
        if avg_cv_std < 0.035:
            print("✅ Excellent model stability")
        elif avg_cv_std < 0.06:
            print("✅ Good model stability")
        else:
            print("⚠️ Model stability needs improvement")
        
        if avg_total_r2 >= 0.75:
            print("✅ Excellent predictive performance")
        elif avg_total_r2 >= 0.65:
            print("✅ Good predictive performance")
        elif avg_total_r2 >= 0.55:
            print("✅ Acceptable predictive performance")
        else:
            print("⚠️ Predictive performance needs improvement")
        
        if avg_mape < 20:
            print("✅ Excellent MAPE")
        elif avg_mape < 25:
            print("✅ Good MAPE")
        else:
            print("⚠️ MAPE needs improvement")
        
        # Check for signs of overfitting or underfitting
        if avg_total_r2 >= 0.75 and avg_total_r2 <= 0.85:
            print("\n✅ Model fit is balanced, with no clear signs of overfitting or underfitting")
        elif avg_total_r2 > 0.85:
            print("\n⚠️ Possible mild overfitting")
        else:
            print("\n⚠️ Possible mild underfitting")
        
        print("✅ Evaluation complete")
        
        return final_imputed_original, eval_results
        
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()
        return None, None

if __name__ == "__main__":
    print("\nStarting the workflow...")
    print("Optimization objectives:")
    print("1. Balance overfitting and underfitting")
    print("2. Improve model generalization")
    print("3. Maintain prediction accuracy")
    print("4. Improve result stability")
    
    start_time = time.time()
    result, evaluation = main()
    
    if result is not None:
        print(f"\n✅ Run complete!")
        print(f"Total time: {(time.time() - start_time)/60:.2f} minutes")
        
        # Performance summary
        if evaluation is not None:
            print("\n" + "=" * 50)
            print("Performance summary")
            print("=" * 50)
            
            avg_total_r2 = evaluation['Overall R²'].mean()
            avg_cv_r2 = evaluation['CV Mean R²'].mean()
            avg_mape = evaluation['MAPE(%)'].mean()
            
            print(f"Final mean overall R²: {avg_total_r2:.4f}")
            print(f"Final mean CV R²: {avg_cv_r2:.4f}")
            print(f"Final mean MAPE: {avg_mape:.2f}%")
            print(f"R² range: [{evaluation['Overall R²'].min():.4f}, {evaluation['Overall R²'].max():.4f}]")
            
            print("\n🎉 Model optimization complete!")
            print("Balanced model settings applied.")
            
            # Suggestions for improvement
            if avg_total_r2 < 0.75:
                print("\nConsider adding data or improving the feature engineering to raise performance.")
            else:
                print("\nThe model meets the configured performance threshold")
            
            print("\nThank you for using this notebook!")
    else:
        print("\n❌ Run failed. See the error above.")
    
    gc.collect()

# ===========================================================
# Entry point
# ===========================================================
if __name__ == "__main__":
    print("\nStarting the workflow...")
    start_time = time.time()
    result, evaluation = main()
    if result is not None:
        print("\n✅ Observed and predicted values for all features were saved to CSV files.")


TensorFlow version: 2.21.0
No GPU found. Running on the CPU.


Starting the workflow...
Optimization objectives:
1. Balance overfitting and underfitting
2. Improve model generalization
3. Maintain prediction accuracy
4. Improve result stability
High-performance VAE imputation - balanced optimization
Main improvements:
1. Balanced regularization
2. Optimized model complexity
3. Combined loss function
4. Balanced ensemble weights
5. Improved evaluation metrics

1. Loading data...

Error: [Errno 2] No such file or directory: './Earthquake.csv'

❌ Run failed. See the error above.

Starting the workflow...
High-performance VAE imputation - balanced optimization
Main improvements:
1. Balanced regularization
2. Optimized model complexity
3. Combined loss function
4. Balanced ensemble weights
5. Improved evaluation metrics

1. Loading data...

Error: [Errno 2] No such file or directory: './Earthquake.csv'


Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23856\202605219.py", line 854, in main
    data_raw, original_data = load_and_preprocess_data()
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23856\202605219.py", line 64, in load_and_preprocess_data
    data = pd.read_csv("./Earthquake.csv", low_memory=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line

In [2]:
import pandas as pd

def add_L_column_to_results():
    """
    Add the L(m) column to the imputed results.
    """
    # 1. Read the source data to retrieve L(m)
    original_data = pd.read_csv("./Earthquake.csv", low_memory=True)
    
    # 2. Check whether L(m) is present
    if 'L(m)' not in original_data.columns:
        print("The source data does not contain L(m)")
        return
    
    # 3. Extract L(m)
    L_column = original_data['L(m)'].copy()
    print(f"Extracted L(m) with {len(L_column)} rows")
    
    # 4. Read the imputed results
    try:
        imputed_results = pd.read_csv("imputed_earthquake.csv")
        print(f"Imputed data shape: {imputed_results.shape}")
    except FileNotFoundError:
        print("Imputed results file not found. Run the imputation cell first.")
        return
    
    # 5. Check that the row counts match
    if len(imputed_results) != len(L_column):
        print(f"Warning: imputed results have {len(imputed_results)} rows, while L(m) has {len(L_column)} rows")
        # If row counts differ, align both tables to the shorter length
        min_rows = min(len(imputed_results), len(L_column))
        imputed_results = imputed_results.iloc[:min_rows]
        L_column = L_column.iloc[:min_rows]
    
    # 6. Add L(m) to the results
    imputed_results['L(m)'] = L_column.values
    
    # 7. Save the updated results
    output_filename = "imputed_earthquake_with_runout.csv"
    imputed_results.to_csv(output_filename, index=False)
    
    print(f"\nAdded L(m) to the imputed results.")
    print(f"Saved the new file as: {output_filename}")
    print(f"Final data shape: {imputed_results.shape}")
    print(f"Column names: {list(imputed_results.columns)}")
    
    # 8. Report summary statistics for L(m)
    print("\nL(m) summary statistics:")
    print(f"  Data type: {L_column.dtype}")
    print(f"  Unique values: {L_column.nunique()}")
    print(f"  Missing values: {L_column.isna().sum()}")
    if pd.api.types.is_numeric_dtype(L_column):
        print(f"  Minimum: {L_column.min()}")
        print(f"  Maximum: {L_column.max()}")
        print(f"  Mean: {L_column.mean():.2f}")
    else:
        print(f"  First five values: {L_column.head().tolist()}")
    
    return imputed_results

# Run the function
if __name__ == "__main__":
    result_with_L = add_L_column_to_results()

Extracted L(m) with 4330 rows
Imputed data shape: (4330, 3)

Added L(m) to the imputed results.
Saved the new file as: imputed_earthquake_with_runout.csv
Final data shape: (4330, 4)
Column names: ['H(m)', 'At(m²）', 'Wa(m)', 'L(m)']

L(m) summary statistics:
  Data type: float64
  Unique values: 1141
  Missing values: 0
  Minimum: 0.26
  Maximum: 44000.0
  Mean: 279.66
